# Databricks Cluster Management: Step-by-Step Implementation Guide
### Interactive Floating Window, Logout / Session End, and Safe Dropdown Hydration

---

## 1. What We Are Implementing in Your Codebase

Your current code works, and we are adding **3 specific enterprise features** from this repository into your app:

1. **No Automatic Cluster Restart $\rightarrow$ Floating Confirmation Modal**:
   * If the app is idle and the all-purpose cluster terminates, the app will **NOT** automatically restart the cluster.
   * Instead, it displays a clean floating modal asking the user: *"Databricks Spark Cluster is Offline. Would you like to start the cluster to load data and use the app?"* with `[Yes, Start Cluster]` and `[Cancel]` buttons.
   * Only when the user confirms does it initiate startup and display the *'Starting Databricks Cluster (2-4 min)...'* spinner modal.

2. **Log Out & End Session Floating Window**:
   * Adds a **Log Out** button to the top header.
   * Clicking it opens a dual-option modal:
     * **End App Session**: Locks the screen, pauses background cluster polling, and shows a *'Resume Session'* button.
     * **Log Out of Databricks**: Signs out of the Databricks workspace completely (`/login.html`).
     * **Cancel**: Returns to the app.

3. **Safe Dropdown Loading (No Crashes When Cluster is Offline)**:
   * When the cluster is terminated or starting, `dropdown_load.py` does **NOT** try to query `gas_shiny_input_versions`.
   * Dropdowns show a clean placeholder: *"Cluster offline. Click Start Cluster above"*.
   * As soon as the user starts the cluster and it reaches `RUNNING`, the app **automatically loads the tables and populates the dropdowns** without refreshing the page.

## 2. Implementation Overview (File by File)

```
my_pyshiny_app/
│
├── cluster_manager.py     <-- ADD modals (offline prompt, spinner, logout, session end) & reactive handlers
│                              (KEEP your existing get_workspace_client, get_spark_session, etc.)
│
├── server_main.py         <-- REMOVE auto-start lines; ADD register_cluster_handlers & pass is_cluster_ready
│                              (KEEP all your bulk_upload, download_template, etc. unchanged)
│
├── dropdown_load.py       <-- WRAP table loading inside @reactive.event(is_cluster_ready)
│                              (KEEP all your metric configuration and dropdown processing unchanged)
│
├── ui_main.py             <-- ADD ui.output_ui("cluster_banner") and Logout button to header
│                              (KEEP all your tabs and visual components unchanged)
│
└── www/custom.css         <-- ADD modal and banner styling
```

## 3. FILE 1: `cluster_manager.py`
**Location**: `./cluster_manager.py`

### 🟢 WHAT TO KEEP (DO NOT TOUCH):
* Keep your `ClusterStatus(Enum)` class.
* Keep your `get_workspace_client()` function.
* Keep your `get_cluster_id()` function.
* Keep your `get_cached_status()` and `get_cluster_status()` functions.
* Keep your `start_cluster()` function.
* Keep your `get_spark_session()` and `reset_spark_session()` functions.
* Keep your `is_session_invalid_error()` function.

### 🟡 WHAT TO ADD (Append this to the bottom of `cluster_manager.py`):
This adds:
1. `cluster_offline_modal()`: The floating window prompting user confirmation.
2. `cluster_starting_modal()`: The spinner modal.
3. `logout_confirmation_modal()` & `session_ended_modal()`: The logout dialogs.
4. `register_cluster_handlers()`: Manages user inactivity, polling, and modal interactions.

In [ ]:
# ===========================================================================
# APPEND THIS TO THE BOTTOM OF cluster_manager.py
# ===========================================================================

import asyncio
from shiny import Inputs, Outputs, Session, reactive, render, ui

# ---------------------------------------------------------------------------
# 1. Floating Modals (Offline Prompt, Starting Spinner, Logout & Session)
# ---------------------------------------------------------------------------

def cluster_offline_modal(cluster_name: str = "", state: str = "TERMINATED") -> ui.Tag:
    """Floating window asking the user whether to start the cluster."""
    display_name = cluster_name or "Databricks Spark Cluster"
    return ui.modal(
        ui.div(
            ui.div(
                ui.HTML('''
                    <svg width="36" height="36" viewBox="0 0 24 24" fill="none" stroke="#e07000" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round">
                        <rect x="2" y="2" width="20" height="8" rx="2" ry="2"></rect>
                        <rect x="2" y="14" width="20" height="8" rx="2" ry="2"></rect>
                        <line x1="6" y1="6" x2="6.01" y2="6"></line>
                        <line x1="6" y1="18" x2="6.01" y2="18"></line>
                    </svg>
                '''),
                class_="status-modal-icon-circle cluster-offline",
                style="margin-bottom: 12px;"
            ),
            ui.div("Databricks Cluster is Offline", class_="status-modal-title", style="font-size: 1.35rem; font-weight: 800; margin-bottom: 8px;"),
            ui.div(
                f"The cluster '{display_name}' has stopped due to inactivity (status: {state.lower()}). "
                "Would you like to start the cluster now to load data, populate dropdowns, and run calculations?",
                class_="status-modal-subtitle",
                style="color: #475569; font-size: 0.92rem; margin-bottom: 20px; max-width: 440px; margin-left: auto; margin-right: auto; line-height: 1.5;"
            ),
            ui.div(
                ui.input_action_button(
                    "btn_start_cluster_confirm",
                    "Yes, Start Cluster",
                    class_="btn btn-primary",
                    style="min-width: 160px; font-weight: 700; background-color: #005eb8; border-color: #005eb8; padding: 9px 20px;"
                ),
                ui.input_action_button(
                    "btn_offline_modal_dismiss",
                    "Cancel",
                    class_="btn btn-outline-secondary",
                    style="min-width: 100px; padding: 9px 20px;"
                ),
                style="display: flex; justify-content: center; gap: 12px; margin-top: 10px;"
            ),
            class_="status-modal-content",
            style="text-align: center; padding: 24px 16px;"
        ),
        title=None,
        footer=None,
        easy_close=False,
        size="m"
    )

def cluster_starting_modal(cluster_name: str = "") -> ui.Tag:
    """Spinner modal shown while compute is booting up."""
    display_name = f" '{cluster_name}'" if cluster_name else ""
    return ui.modal(
        ui.div(
            ui.div(class_="spinner-border text-primary", style="width: 3rem; height: 3rem; color: #005eb8; margin-bottom: 15px;"),
            ui.div(f"Starting Databricks Cluster{display_name}...", class_="status-modal-title", style="font-size: 1.35rem; font-weight: 800; margin-bottom: 8px;"),
            ui.div(
                "Startup has been requested in Databricks. It usually takes 2 to 4 minutes. "
                "The app will automatically connect and load dropdowns once the cluster is online.",
                class_="status-modal-subtitle",
                style="color: #475569; font-size: 0.92rem; margin-bottom: 20px; max-width: 440px; margin-left: auto; margin-right: auto; line-height: 1.5;"
            ),
            ui.div(
                ui.input_action_button("btn_dismiss_starting_modal", "Wait in Background", class_="btn btn-outline-secondary btn-sm"),
                style="display: flex; justify-content: center;"
            ),
            class_="status-modal-content",
            style="text-align: center; padding: 24px 16px;"
        ),
        title=None,
        footer=None,
        easy_close=False,
        size="m"
    )

def logout_confirmation_modal(user_email: str = "", databricks_host: str = "") -> ui.Tag:
    """Logout dialog giving user choice between ending session or logging out of Databricks."""
    clean_host = databricks_host if databricks_host.startswith("http") else f"https://{databricks_host}" if databricks_host else ""
    logout_url = f"{clean_host.rstrip('/')}/login.html" if clean_host else "/login.html"
    
    user_badge = ui.div(
        ui.div("SIGNED IN AS", style="font-size: 0.72rem; font-weight: 700; color: #64748b; letter-spacing: 0.05em;"),
        ui.div(user_email or "Databricks User", style="font-size: 1rem; font-weight: 600; color: #1e293b;"),
        style="background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 8px; padding: 8px 16px; margin-bottom: 20px; text-align: center;"
    )

    return ui.modal(
        ui.div(
            ui.div("Log Out or End Your Session", style="font-size: 1.35rem; font-weight: 800; color: #0f172a; margin-bottom: 6px; text-align: center;"),
            ui.div("Choose to pause your active application session or sign out of your Databricks workspace completely.", style="font-size: 0.9rem; color: #64748b; text-align: center; margin-bottom: 18px;"),
            user_badge,
            ui.div(
                # Option 1: End App Session
                ui.div(
                    ui.div("⏸️ End App Session", style="font-weight: 700; color: #1e293b; font-size: 0.95rem; margin-bottom: 4px;"),
                    ui.div("Pause background compute polling and lock your session without losing login credentials.", style="font-size: 0.8rem; color: #64748b; margin-bottom: 12px; flex-grow: 1;"),
                    ui.input_action_button("btn_confirm_end_session", "End Session", class_="btn btn-primary btn-sm", style="width: 100%; font-weight: 700; background-color: #005eb8;"),
                    style="flex: 1; border: 1px solid #e2e8f0; border-radius: 10px; padding: 16px; display: flex; flex-direction: column; background: #fff;"
                ),
                # Option 2: Log Out of Databricks
                ui.div(
                    ui.div("🚪 Log Out of Databricks", style="font-weight: 700; color: #1e293b; font-size: 0.95rem; margin-bottom: 4px;"),
                    ui.div("Sign out completely from your Databricks workspace and close this application.", style="font-size: 0.8rem; color: #64748b; margin-bottom: 12px; flex-grow: 1;"),
                    ui.tags.a("Log Out", href=logout_url, target="_top", class_="btn btn-outline-danger btn-sm", style="width: 100%; font-weight: 700; text-align: center; text-decoration: none;"),
                    style="flex: 1; border: 1px solid #e2e8f0; border-radius: 10px; padding: 16px; display: flex; flex-direction: column; background: #fff;"
                ),
                style="display: flex; gap: 14px; margin-bottom: 18px;"
            ),
            ui.div(
                ui.input_action_button("btn_dismiss_logout_modal", "Cancel", class_="btn btn-outline-secondary", style="min-width: 100px;"),
                style="display: flex; justify-content: center;"
            ),
            style="padding: 10px 8px;"
        ),
        title=None,
        footer=None,
        easy_close=True,
        size="m"
    )

def session_ended_modal() -> ui.Tag:
    """Locked session screen shown when user clicks End Session."""
    return ui.modal(
        ui.div(
            ui.div("Session Ended", style="font-size: 1.35rem; font-weight: 800; color: #0f172a; margin-bottom: 8px; text-align: center;"),
            ui.div("Your session has been locked and background compute polling is paused.", style="font-size: 0.92rem; color: #64748b; margin-bottom: 20px; text-align: center;"),
            ui.div(
                ui.input_action_button("btn_resume_session", "Resume Session", class_="btn btn-primary", style="min-width: 160px; font-weight: 700; background-color: #005eb8;"),
                style="display: flex; justify-content: center;"
            ),
            style="padding: 24px 16px; text-align: center;"
        ),
        title=None,
        footer=None,
        easy_close=False,
        size="m"
    )

def render_cluster_banner(status: ClusterStatus) -> ui.Tag | None:
    """Top alert banner. Automatically returns None (hidden) when RUNNING!"""
    if status in (ClusterStatus.RUNNING, ClusterStatus.RESIZING):
        return None

    is_offline = status in (ClusterStatus.TERMINATED, ClusterStatus.ERROR, ClusterStatus.UNKNOWN)
    is_starting = status in (ClusterStatus.STARTING, ClusterStatus.PENDING, ClusterStatus.RESTARTING)

    if is_offline:
        action_btn = ui.input_action_button(
            "btn_banner_start_cluster",
            "Start Cluster",
            class_="btn btn-primary btn-sm",
            style="margin-left: auto; font-weight: 700; background-color: #005eb8; border-color: #005eb8;"
        )
    elif is_starting:
        action_btn = ui.div(
            ui.tags.span(class_="spinner-border spinner-border-sm text-primary", style="margin-right: 6px;"),
            ui.tags.span("Starting Spark cluster (2-4 min)..."),
            style="margin-left: auto; display: flex; align-items: center; font-size: 0.85rem; font-weight: 700; color: #005eb8;"
        )
    else:
        action_btn = None

    return ui.div(
        ui.tags.span(f"Spark Cluster: {status.value.upper()}", style="font-weight: 800; margin-right: 14px;"),
        ui.tags.span("Compute is stopped. Start cluster to load dropdowns and run PySpark calculations."),
        action_btn,
        style="display: flex; align-items: center; padding: 10px 20px; font-size: 0.85rem; border-bottom: 1px solid #fed7aa; background-color: #fff7ed; color: #9a3412; margin-bottom: 15px;"
    )


# ---------------------------------------------------------------------------
# 2. Reactive Handler Coordinator (Handles Offline Prompt, Start, & Logout)
# ---------------------------------------------------------------------------

def register_cluster_handlers(input: Inputs, output: Outputs, session: Session) -> reactive.Value[bool]:
    """
    Manages cluster polling, floating modals, and the Logout button.
    DOES NOT AUTO-START CLUSTER: Shows floating confirmation modal to user.
    RETURNS: is_cluster_ready (reactive.Value[bool])
    """
    initial_status = get_cluster_status()
    is_initial_ready = initial_status in (ClusterStatus.RUNNING, ClusterStatus.RESIZING)

    cluster_status_val = reactive.value(initial_status)
    is_cluster_ready = reactive.value(is_initial_ready)
    modal_state = reactive.value("none")  # 'none', 'offline_prompt', 'starting', 'session_ended'
    offline_modal_dismissed = reactive.value(False)

    # Render top banner
    @output
    @render.ui
    def cluster_banner():
        return render_cluster_banner(cluster_status_val())

    # Non-blocking status poller
    @reactive.effect
    async def poll_cluster_status_loop():
        if modal_state() == "session_ended":
            return  # Polling paused while session is ended

        status = await asyncio.to_thread(get_cluster_status, force_refresh=True)
        cluster_status_val.set(status)
        ready = status in (ClusterStatus.RUNNING, ClusterStatus.RESIZING)

        if is_cluster_ready() != ready:
            is_cluster_ready.set(ready)

        # CASE 1: CLUSTER IS RUNNING
        if ready:
            if modal_state() in {"starting", "offline_prompt"}:
                modal_state.set("none")
                offline_modal_dismissed.set(False)
                try:
                    ui.modal_remove()
                except Exception:
                    pass
                ui.notification_show("Spark cluster is online! Loading dropdowns and data...", type="message", duration=5)
            reactive.invalidate_later(20)  # Check every 20s while running
            return

        # CASE 2: CLUSTER IS STARTING / PENDING
        if status in (ClusterStatus.STARTING, ClusterStatus.PENDING, ClusterStatus.RESTARTING):
            reactive.invalidate_later(3)  # Check every 3s while booting
            return

        # CASE 3: CLUSTER IS TERMINATED / OFFLINE
        # Instead of auto-starting, open floating window asking user if they want to start it!
        if not offline_modal_dismissed() and modal_state() == "none":
            modal_state.set("offline_prompt")
            ui.modal_show(cluster_offline_modal(state=status.value))

        reactive.invalidate_later(30)  # Check every 30s while offline

    # User clicks 'Yes, Start Cluster' in confirmation modal
    @reactive.effect
    @reactive.event(input.btn_start_cluster_confirm)
    async def on_start_cluster_confirmed():
        modal_state.set("starting")
        ui.modal_show(cluster_starting_modal())
        try:
            print("[ClusterManager] User confirmed start. Calling start_cluster()...")
            await asyncio.to_thread(start_cluster)
        except Exception as e:
            ui.notification_show(f"Failed to start cluster: {e}", type="error", duration=10)

    # User clicks 'Start Cluster' on top banner
    @reactive.effect
    @reactive.event(input.btn_banner_start_cluster)
    def on_banner_start_cluster():
        offline_modal_dismissed.set(False)
        modal_state.set("offline_prompt")
        ui.modal_show(cluster_offline_modal(state=cluster_status_val().value))

    # User cancels offline prompt
    @reactive.effect
    @reactive.event(input.btn_offline_modal_dismiss)
    def on_offline_modal_dismiss():
        offline_modal_dismissed.set(True)
        modal_state.set("none")
        ui.modal_remove()

    @reactive.effect
    @reactive.event(input.btn_dismiss_starting_modal)
    def on_dismiss_starting_modal():
        modal_state.set("none")
        ui.modal_remove()

    # -----------------------------------------------------------------------
    # Logout & Session Management Event Handlers
    # -----------------------------------------------------------------------
    @reactive.effect
    @reactive.event(input.btn_logout_prompt)
    def on_logout_prompt():
        host = os.environ.get("DATABRICKS_HOST", "")
        user_email = os.environ.get("DATABRICKS_USER_EMAIL", "Databricks User")
        ui.modal_show(logout_confirmation_modal(user_email=user_email, databricks_host=host))

    @reactive.effect
    @reactive.event(input.btn_dismiss_logout_modal)
    def on_dismiss_logout_modal():
        ui.modal_remove()

    @reactive.effect
    @reactive.event(input.btn_confirm_end_session)
    def on_confirm_end_session():
        modal_state.set("session_ended")
        ui.modal_show(session_ended_modal())

    @reactive.effect
    @reactive.event(input.btn_resume_session)
    def on_resume_session():
        modal_state.set("none")
        ui.modal_remove()
        # Trigger immediate status check
        status = get_cluster_status(force_refresh=True)
        cluster_status_val.set(status)
        if status not in (ClusterStatus.RUNNING, ClusterStatus.RESIZING):
            modal_state.set("offline_prompt")
            ui.modal_show(cluster_offline_modal(state=status.value))

    return is_cluster_ready

## 4. FILE 2: `server_main.py`
**Location**: `./server_main.py`

### 🔴 WHAT TO REMOVE:
1. **Lines 24–41 (REMOVE THE AUTO-START ON MODULE LOAD)**:
   Remove these lines:
   ```python
   # REMOVE THIS:
   _initial_status = get_cluster_status()
   if _initial_status == ClusterStatus.TERMINATED:
       _start_result = start_cluster()
       _cluster_start_attempted = True
   ```
2. **Lines 53–100 (REMOVE YOUR OLD `poll_cluster_status`)**:
   Remove your old `poll_cluster_status` reactive effect, which automatically started the cluster on lines 77–83:
   ```python
   # REMOVE THIS:
   if status == ClusterStatus.TERMINATED and not _cluster_start_attempted:
       start_result = start_cluster()  # <-- REMOVE AUTO-START!
   ```

### 🟢 WHAT TO KEEP:
* Keep all your module imports (`bulk_upload_inputs`, `download_template`, `scenario_calc`, `scorecard_extract_final`).
* Keep all your submodule function calls.

### 🟡 WHAT TO ADD / REPLACE INSIDE `def server(input, output, session)`:

In [ ]:
# Code for server_main.py

from shiny import ui, render, reactive
import plotly.graph_objects as go
from datetime import datetime, timedelta
import pandas as pd

# Import cluster management handlers
from cluster_manager import register_cluster_handlers

# Import your existing server modules (KEEP ALL UNCHANGED):
from server_scripts.visuals.visuals_main import visuals_main
from server_scripts.inputs.bulk_upload_validation import bulk_upload_inputs
from server_scripts.output.download_template_calculations import download_template
from server_scripts.calculations.scenario_comparison_calculations_main import scenario_calc
from server_scripts.output.scorecard_extract import scorecard_extract_final


def server(input, output, session):
    # =========================================================================
    # 1. Cluster Status & Floating Window Management
    # =========================================================================
    # When the cluster terminates, this automatically displays the floating window
    # asking the user whether to start the cluster. It will NOT auto-start.
    # It also handles the 'Log Out' button and 'End Session' modal.
    is_cluster_ready = register_cluster_handlers(input, output, session)

    # =========================================================================
    # 2. Main Application Logic
    # =========================================================================
    # Pass is_cluster_ready to visuals_main so dropdowns wait for cluster startup:
    visuals_main(input, output, session, is_cluster_ready)

    # Keep your other submodules exactly as they are:
    bulk_upload_inputs(input, output, session)
    download_template(input, output, session)
    scenario_calc(input, output, session)
    scorecard_extract_final(input, output, session)

## 5. FILE 3: `dropdown_load.py` (Wait for Cluster to Start)
**Location**: `./server_scripts/visuals/dropdown_load.py`

This is from your screenshot `IMG_20261109_142808596_HDR.jpg`.

### 🔴 WHAT TO REMOVE:
Remove lines 24–30:
```python
# REMOVE THIS BLOCK:
_initial_status = get_cluster_status()
if _initial_status == ClusterStatus.TERMINATED:
    start_cluster()
    return print("Cluster not running")
```

### 🟢 WHAT TO KEEP:
* Keep your `dropdown_remarks_config` dictionary.
* Keep your metric loop: `for metric, config in dropdown_remarks_config.items(): ...`
* Keep `dropdown_update`, `delete_dropdown`, and `undo_dropdown`.

### 🟡 WHAT TO REPLACE IT WITH:
Wrap the table loading in `@reactive.effect @reactive.event(is_cluster_ready_signal)`.
* If the cluster is offline, dropdowns show: *"Cluster offline. Click Start Cluster above"* (no crash!).
* Once the cluster starts, this function automatically executes, reads `gas_shiny_input_versions` and `gas_shiny_scenario_log`, and populates all dropdowns!

In [ ]:
# Code for dropdown_load.py (or load_input.py)

from shiny import ui, reactive, render
from common_functions import load_delta_table
from cluster_manager import is_cluster_ready

def final_load_dropdown_remarks(dropdown_remarks_config, input, session, is_cluster_ready_signal):
    """
    Populates dropdowns from Databricks tables.
    If the cluster is offline, it safely waits without failing.
    When the cluster reaches RUNNING, it loads tables and updates dropdowns automatically.
    """
    
    @reactive.effect
    @reactive.event(is_cluster_ready_signal)
    def _populate_dropdowns_when_ready():
        # 1. Cluster is NOT ready yet: Set safe placeholder in dropdowns
        if not is_cluster_ready_signal():
            print("[dropdown_load] Cluster is offline. Setting safe placeholder in dropdowns...")
            for metric, config in dropdown_remarks_config.items():
                d_id = config.get("dropdown_id")
                if d_id:
                    ui.update_select(
                        d_id, 
                        choices=["Cluster offline. Start cluster above to load options"],
                        selected="Cluster offline. Start cluster above to load options"
                    )
            return

        # 2. Cluster is RUNNING! Safe to load tables:
        try:
            print("[dropdown_load] Cluster is RUNNING! Loading gas_shiny_input_versions & gas_shiny_scenario_log...")
            df1 = load_delta_table("gas_shiny_input_versions")
            df2 = load_delta_table("gas_shiny_scenario_log")
            
            # =========================================================
            # YOUR EXISTING DROPDOWN PROCESSING CODE RUNS HERE UNCHANGED:
            # =========================================================
            for metric, config in dropdown_remarks_config.items():
                dropdown_id = config["dropdown_id"]
                delete_id = config["delete_id"]
                
                # Extract available versions from df1/df2 as you normally do:
                if "version" in df1.columns:
                    available_versions = df1["version"].dropna().unique().tolist()
                    if available_versions:
                        ui.update_select(dropdown_id, choices=available_versions, selected=available_versions[0])
                        
            ui.notification_show("Dropdown options successfully loaded from Databricks!", type="message", duration=4)
        except Exception as e:
            print(f"[dropdown_load] Error populating dropdowns: {e}")
            ui.notification_show(f"Error loading dropdowns: {e}", type="warning", duration=6)

## 6. FILE 4: `ui_main.py` (Add Log Out Button & Cluster Banner)
**Location**: `./ui_main.py`

### 🟢 WHAT TO KEEP:
* Keep all your navigation tabs, layout sidebars, and panels.

### 🟡 WHAT TO ADD:
1. Add `ui.output_ui("cluster_banner")` at the very top.
2. Add the **Log Out** button: `ui.input_action_button("btn_logout_prompt", "Log Out", ...)` in your header.

In [ ]:
# Code to update in ui_main.py

from shiny import ui

def ui_main():
    return ui.page_fluid(
        # 1. Custom CSS styles for banner and modals
        ui.include_css("www/custom.css"),
        
        # 2. TOP STATUS BANNER (Auto-hides when RUNNING, shows when STOPPED)
        ui.output_ui("cluster_banner"),
        
        # 3. HEADER WITH LOG OUT BUTTON
        ui.div(
            ui.div(
                ui.h4("Capacity Planning & Forecasting App", style="font-weight: 800; color: #005eb8; margin: 0;"),
                style="display: flex; align-items: center;"
            ),
            ui.div(
                ui.input_action_button(
                    "btn_logout_prompt",
                    "Log Out",
                    class_="btn btn-outline-danger btn-sm",
                    style="font-weight: 700; padding: 6px 16px; border-radius: 6px;"
                ),
                style="display: flex; align-items: center; margin-left: auto;"
            ),
            style="display: flex; align-items: center; justify-content: space-between; padding: 12px 0 16px 0; border-bottom: 1px solid #e2e8f0; margin-bottom: 18px;"
        ),
        
        # (YOUR EXISTING TABS / VIEWS CONTINUE HERE UNCHANGED)
        ui.navset_tab(
            # ... your existing tabs ...
            id="main_tabset"
        )
    )

## 7. How It Behaves in Real Life (The User Experience)

| Scenario | What Happens in the App |
| :--- | :--- |
| **User opens app when cluster is terminated** | App loads immediately with zero errors. Dropdowns display *'Cluster offline. Start cluster above'*. A floating window opens asking: *'Databricks Cluster is Offline. Would you like to start the cluster?'* with `[Yes, Start Cluster]` and `[Cancel]`. |
| **User clicks 'Cancel'** | The modal closes. The app remains usable for browsing offline pages/reports. The top banner stays visible with a `[Start Cluster]` button if they change their mind. |
| **User clicks 'Yes, Start Cluster'** | The modal turns into the *'Starting Databricks Cluster (2-4 min)...'* spinner dialog. The poller checks every 3 seconds in the background. |
| **Cluster reaches RUNNING state** | The spinner modal auto-closes. The top banner vanishes. A toast appears: *'Spark cluster is online! Loading dropdowns and data...'*. `dropdown_load.py` reads `gas_shiny_input_versions` and populates the dropdowns automatically! |
| **Cluster auto-terminates after idle timeout** | The app detects the state changed to `TERMINATED`. Instead of auto-restarting, it opens the floating confirmation modal asking the user if they are still working and want to restart it. |
| **User clicks 'Log Out' button** | Opens the *'Log Out or End Your Session'* dialog. The user can either pause their session (`End App Session`) or completely sign out of Databricks (`Log Out of Databricks`). |